# 🏆 Leaderboard · Task 2 · MovieLens top-10

Starter for Task 2 of the [Leaderboard](Leaderboard.md): recommend 10 movies per user, scored by **full-ranking NDCG@10** (higher is better).

| | |
|---|---|
| **Data** | MovieLens `ml-latest-small` (100 836 ratings, 610 users, 9 742 movies), the dataset of [[Project - MovieLens Recommender]] |
| **Split** | the project's split: ratings ≥ 4 are positives; per user, the latest positive is **test**, the one before is **validation**, the rest is **train** (609 test users) |
| **Metric** | NDCG@10 of the single test movie, ranked against **all** 9 742 movies minus the user's train + validation positives; ties broken by movie id |
| **Baseline** | most popular movies (count of positives in train + validation): **0.0217** |
| **Rules** | the model may use train positives (and validation positives for the final refit) plus `movies.csv`/`links.csv` metadata · no ratings below 4, no test rows · tune on validation |

Why full ranking and not the project's "1 positive vs 99 sampled negatives"? Sampled metrics are noisier and can reorder models (see [[Evaluating Recommenders]]). Full-ranking numbers are much smaller: 0.02 here is normal, not broken.


**How to use this notebook**
1. Run everything once as delivered: it downloads the data, checks the split is the official one, and reproduces the baseline.
2. Write your model in **`my_model`** (section 4) and set `METHOD` to a short description.
3. Iterate on the **validation** score (section 5) as often as you like.
4. When you are happy, run section 6 **once**: it trains on the full training data, scores the test set, and prints a row for [Leaderboard](Leaderboard.md).
5. Save a copy of the notebook for your attempt if you want the row's link to point to the exact code (*File → Save as*, e.g. `Leaderboard - MNIST - small CNN.ipynb`).

Cells print ✅ / ❌ / ⏳ instead of raising, so the notebook always runs top to bottom. Saved without outputs.

In [ ]:
import datetime, hashlib, pathlib, re, time, urllib.request
import numpy as np

DATA = pathlib.Path("leaderboard-data")   # downloads are cached here (git-ignored)
DATA.mkdir(exist_ok=True)

def check(ok, msg_ok, msg_bad):
    print(("✅ " + msg_ok) if ok else ("❌ " + msg_bad))

def download(url, name):
    f = DATA / name
    if not f.exists():
        print("downloading", url)
        urllib.request.urlretrieve(url, f)
    return f

def leaderboard_best(task, higher_is_better, path="Leaderboard.md"):
    """Best score logged so far in the Leaderboard.md section whose heading contains `task`."""
    p = pathlib.Path(path)
    if not p.exists():
        return None
    best = None
    for sec in re.split(r"^## ", p.read_text(encoding="utf-8"), flags=re.M)[1:]:
        if task not in sec.splitlines()[0]:
            continue
        for line in sec.splitlines():
            cells = [c.strip() for c in line.strip().strip("|").split("|")]
            if len(cells) < 3 or not re.fullmatch(r"\d{4}-\d{2}-\d{2}", cells[0]):
                continue
            try:
                score = float(cells[1])
            except ValueError:
                continue
            if best is None or (score > best[0] if higher_is_better else score < best[0]):
                best = (score, cells[2])
    return best

def report(task, score, method, baseline, higher_is_better, notebook, digits=4):
    """Compare a test score with the baseline and your best so far, and print a row for Leaderboard.md."""
    better = (lambda a, b: a > b) if higher_is_better else (lambda a, b: a < b)
    print(f"Test score: {score:.{digits}f}")
    check(better(score, baseline), f"beats the baseline ({baseline})", f"does not beat the baseline ({baseline}) yet")
    best = leaderboard_best(task, higher_is_better)
    if best:
        check(better(score, best[0]), f"new personal best! Previous best: {best[0]} ({best[1]})",
              f"not a new best. Current best: {best[0]} ({best[1]})")
    print("\nPaste this row at the bottom of the table in Leaderboard.md:")
    print(f"| {datetime.date.today()} | {score:.{digits}f} | {method} | [notebook]({notebook.replace(' ', '%20')}) |")

import pandas as pd, zipfile

if not (DATA / "ml-latest-small").exists():
    zipfile.ZipFile(download("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip",
                             "ml-latest-small.zip")).extractall(DATA)
ratings = pd.read_csv(DATA / "ml-latest-small" / "ratings.csv")   # userId, movieId, rating, timestamp
movies = pd.read_csv(DATA / "ml-latest-small" / "movies.csv")     # movieId, title, genres
links = pd.read_csv(DATA / "ml-latest-small" / "links.csv")       # movieId, imdbId, tmdbId
print(len(ratings), "ratings,", ratings.userId.nunique(), "users,", len(movies), "movies")

## 1. The fixed split
Identical to the MovieLens project notebooks (M1–M5), so a model from the project can be dropped straight in. The fingerprint checks it.

In [ ]:
pos = ratings[ratings.rating >= 4].sort_values(["userId", "timestamp"])
pos["rank"] = pos.groupby("userId").cumcount(ascending=False)   # 0 = most recent
test  = pos[pos["rank"] == 0][["userId", "movieId"]]
val   = pos[pos["rank"] == 1][["userId", "movieId"]]
train = pos[pos["rank"] >= 2][["userId", "movieId"]]
train_val = pd.concat([train, val])

ITEMS = np.array(sorted(movies.movieId.unique()))   # the score vector is indexed by this order
COL = {m: i for i, m in enumerate(ITEMS)}

fingerprint = lambda df: hashlib.sha256(df.sort_values(["userId", "movieId"]).to_numpy().tobytes()).hexdigest()[:12]
print(len(train), "train,", len(val), "validation,", len(test), "test positives")
check(fingerprint(test) == "daf3f3867580" and fingerprint(val) == "b9c952e732b0",
      "split is the official one", "unexpected split: scores are not comparable")

## 2. The official metric
A model is a function `score_fn(user_id) -> np.array` of length 9 742 with one score per movie in `ITEMS` order (higher = recommend first).
The metric hides the movies the user is already known to like, takes the top 10 and checks where the held-out movie landed.

In [ ]:
def ndcg_at_10(score_fn, held_out, known, k=10):
    """Mean NDCG@k of one held-out movie per user, full ranking over ITEMS minus the user's `known` movies.
    Ties are broken by position in ITEMS (stable sort), so the metric is deterministic."""
    known = known.groupby("userId").movieId.apply(lambda s: [COL[m] for m in s]).to_dict()
    gains = []
    for u, m in held_out.itertuples(index=False):
        s = np.array(score_fn(u), dtype=float)
        assert s.shape == (len(ITEMS),), f"score_fn must return {len(ITEMS)} scores, got {s.shape}"
        s[known.get(u, [])] = -np.inf
        top = ITEMS[np.argsort(-s, kind="stable")[:k]]
        hit = np.flatnonzero(top == m)
        gains.append(1 / np.log2(hit[0] + 2) if len(hit) else 0.0)
    return float(np.mean(gains))

def val_ndcg(score_fn):
    """Development score: model fitted on train, evaluated on the validation positive."""
    return ndcg_at_10(score_fn, val, train)

def movielens_ndcg(score_fn):
    """Official Task 2 metric: model fitted on train + validation, evaluated on the test positive."""
    return ndcg_at_10(score_fn, test, train_val)

# Self-test: an oracle that puts the test movie first scores 1, one that puts it second scores 1/log2(3)
truth = dict(test.itertuples(index=False))
known_tv = train_val.groupby("userId").movieId.apply(set).to_dict()
def oracle(rank):
    def score_fn(u):
        s = np.zeros(len(ITEMS)); s[COL[truth[u]]] = 2.0
        if rank == 2:   # put one movie the user hasn't seen above it
            s[next(i for i, m in enumerate(ITEMS) if m != truth[u] and m not in known_tv.get(u, ()))] = 3.0
        return s
    return score_fn
check(movielens_ndcg(oracle(1)) == 1.0 and abs(movielens_ndcg(oracle(2)) - 1 / np.log2(3)) < 1e-12,
      "metric self-test passed", "metric self-test failed")

## 3. Baseline: most popular

In [ ]:
BASELINE = 0.0217

def popularity(fit_df):
    counts = fit_df.movieId.value_counts().reindex(ITEMS).fillna(0).to_numpy()
    return lambda u: counts

print(f"validation NDCG@10 {val_ndcg(popularity(train)):.4f}")
base_ndcg = movielens_ndcg(popularity(train_val))
print(f"test NDCG@10       {base_ndcg:.4f}")
check(round(base_ndcg, 4) == BASELINE, f"baseline reproduced ({BASELINE})", f"baseline differs from {BASELINE}")

## 4. Your model ✏️
`my_model(fit_df)` gets a DataFrame of positives (`userId`, `movieId`) and returns a `score_fn(user_id)` as described in section 2.
You may also read `movies` and `links`. Return `None` until you have something.

⚠️ `score_fn` is called for **every** evaluated user, including users with no positives in `fit_df` (one test user has none at all). Give them a fallback such as popularity instead of crashing.

In [ ]:
METHOD = "my model"   # e.g. "BPR-MF d=64, 30 epochs"

def my_model(fit_df):
    # TODO: fit on fit_df and return score_fn(user_id) -> array of len(ITEMS) scores
    return None

<details>
<summary><b>Example submission</b> (shows the interface; try your own idea first)</summary>

Item–item co-occurrence: score a movie by how often it was liked together with the user's liked movies, divided by its popularity to the power α. Scores **0.0311** test NDCG@10.

```python
METHOD = "item co-occurrence, popularity^0.5 damping"

def my_model(fit_df, alpha=0.5):
    users = {u: i for i, u in enumerate(sorted(ratings.userId.unique()))}   # all users, also cold ones
    X = np.zeros((len(users), len(ITEMS)), dtype=np.float32)
    X[fit_df.userId.map(users), fit_df.movieId.map(COL)] = 1
    pop = X.sum(0)
    C = X.T @ X                                    # co-occurrence counts
    np.fill_diagonal(C, 0)
    C /= np.maximum(pop, 1) ** alpha               # damp very popular movies
    S = X @ C + 1e-6 * pop                         # tiny popularity term: fallback for cold users, tie-breaker otherwise
    return lambda u: S[users[u]]
```

</details>

**Ideas to try:** matrix factorization or BPR from the project milestones (drop them in as `score_fn`), EASE (a closed-form item–item model), recency weighting (recent positives say more about the next one), mixing in popularity.
See [[Collaborative Filtering and Matrix Factorization]], [[Deep Learning Recommenders]], [[Evaluating Recommenders]].

## 5. Validation score (use as often as you like)

In [ ]:
t = time.time()
model = my_model(train)
if model is None:
    print("⏳ my_model returns None: write your model in section 4")
else:
    print(f"validation NDCG@10 {val_ndcg(model):.4f} ({time.time() - t:.0f} s)")

## 6. Test score (once per idea)
Refits on train + validation, scores the test positives and compares with the baseline and with your best row in `Leaderboard.md`.

In [ ]:
model = my_model(train_val)
if model is None:
    print("⏳ my_model returns None: nothing to submit yet")
else:
    report("MovieLens", movielens_ndcg(model), METHOD, BASELINE, higher_is_better=True,
           notebook="Leaderboard - MovieLens.ipynb")